# Causal dual-head training: mBERT (CS6140 Group 5)

XLM-R is already trained by a teammate (XLMR_Update1_Update2 package). This notebook trains **mBERT** with the
**same hyper-parameters and selection rule** (see `streaming/train.py` docstring), so the only difference is the backbone.

**Settings before running:** Accelerator = **GPU** (one T4 or P100 is enough), Internet = **On**
(clones the code from GitHub and downloads `bert-base-multilingual-cased`).
Add the dataset `cs6140-phase1-v4` (the uploaded zip) as input. Only its `data/` part is used; code comes from GitHub.

Run with **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed.
Everything is written to `/kaggle/working/runs`.

Rule: model selection uses **validation only**. Test is scored once at the end.

In [ ]:
import glob, os, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
hits = glob.glob('/kaggle/input/**/labels/train_labels.jsonl', recursive=True)
if not hits:  # dataset stored as a zip: extract it to the working dir
    z = glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]
    print('extracting', z)
    subprocess.run(['unzip', '-q', '-o', z, '-d', '/kaggle/working/phase1'], check=True)
    hits = glob.glob('/kaggle/working/phase1/**/labels/train_labels.jsonl', recursive=True)
DATA = hits[0].rsplit('/labels/', 1)[0]
assert os.path.exists(DATA + '/encoded/mbert/train.pt'), 'mBERT encoding missing from the dataset'
print('DATA =', DATA)
!rm -rf /kaggle/working/cs6140ML && git clone --depth 1 https://github.com/lanselylee/cs6140ML /kaggle/working/cs6140ML
!cd /kaggle/working/cs6140ML && git log -1 --oneline
import transformers, torch; print('torch', torch.__version__, 'transformers', transformers.__version__)

## Train mBERT (3 epochs, micro-batch 4 × accumulation 4, lr 2e-5, seed 6130), then predict + score val and test

In [ ]:
!bash /kaggle/working/cs6140ML/streaming/run_all.sh {DATA} /kaggle/working/runs mbert

## Results

In [ ]:
!cat /kaggle/working/runs/mbert/val_log.csv
!cat /kaggle/working/runs/mbert/meta.json
# keep the downloadable output small: drop the extracted data copy and the 700 MB checkpoint
!rm -rf /kaggle/working/phase1 /kaggle/working/cs6140ML /kaggle/working/runs/mbert/best.pt
!du -sh /kaggle/working/runs/*